In [ ]:
import glob
import cv2
from PIL import Image, ImageDraw
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
path = "/data/video-segmentation/kittistep/images/val/0002"
image_paths = sorted(glob.glob(path + "/*.png"))

In [ ]:
img1 = Image.open(image_paths[0]).convert("RGB")
img2 = Image.open(image_paths[2]).convert("RGB")

In [ ]:
img1.size, img2.size

In [ ]:
img1

In [ ]:
img2

In [ ]:
img1_copy = img1.copy()
draw = ImageDraw.Draw(img1_copy)
r = 5
w1, h1 = 810, 150
draw.ellipse((w1-r, h1-r, w1+r, h1+r), fill="red", outline="white")
w2, h2 = 340, 180
draw.ellipse((w2-r, h2-r, w2+r, h2+r), fill="red", outline="white")
img1_copy

In [ ]:
img1 = cv2.imread(image_paths[0])
img2 = cv2.imread(image_paths[2])

gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

# Initial points to track: shape (N, 1, 2), float32
p0 = np.array([
    [[150, 810]],
    [[180, 340]],
], dtype=np.float32)

p1, status, error = cv2.calcOpticalFlowPyrLK(
    gray1,
    gray2,
    p0,
    None
)

# Keep successfully tracked points
good_old = p0[status.ravel() == 1]
good_new = p1[status.ravel() == 1]

for i in range(len(p0)):
    print(
        f"Point: {p0[i, 0]}, "
        f"Tracked: {p1[i, 0]}, "
        f"Status: {status[i, 0]}, "
        f"Error: {error[i, 0]}"
    )

In [ ]:
p1, status, error = cv2.calcOpticalFlowPyrLK(
    gray1, gray2, p0, None,
    winSize=(21, 21),
    maxLevel=3
)



In [ ]:
good_old, good_new

In [ ]:
lk_params = {
    "winSize": (21, 21),
    "maxLevel": 3,
    "criteria": (
        cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT,
        30,
        0.01
    )
}

out = img2.copy()

for old, new in zip(good_old, good_new):
    x1, y1 = old.ravel().astype(int)
    x2, y2 = new.ravel().astype(int)

    cv2.arrowedLine(out, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.circle(out, (x2, y2), 4, (0, 0, 255), -1)

plt.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
plt.axis("off");

In [ ]:
x1

In [ ]:
img1 = cv2.imread(image_paths[0])
img2 = cv2.imread(image_paths[5])

gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

flow = cv2.calcOpticalFlowFarneback(
    gray1,
    gray2,
    None,
    pyr_scale=0.5,
    levels=3,
    winsize=15,
    iterations=3,
    poly_n=5,
    poly_sigma=1.2,
    flags=0
)

In [ ]:
print(flow.shape)  # (H, W, 2)

In [ ]:
mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])

hsv = np.zeros((*gray1.shape, 3), dtype=np.uint8)
hsv[..., 0] = ang * 180 / np.pi / 2
hsv[..., 1] = 255
hsv[..., 2] = cv2.normalize(
    mag, None, 0, 255, cv2.NORM_MINMAX
)

flow_rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

plt.figure(figsize=(12, 8))
plt.imshow(flow_rgb)
plt.axis("off");

In [ ]:
out = img2.copy()

step = 16
h, w = gray1.shape

for y in range(0, h, step):
    for x in range(0, w, step):
        dx, dy = flow[y, x]

        cv2.arrowedLine(
            out,
            (x, y),
            (int(x + dx), int(y + dy)),
            (0, 255, 0),
            1
        )

plt.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
plt.axis("off");